In [ ]:
# import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

In [ ]:
# connection
from snowflake.snowpark.context import get_active_session
session = get_active_session()
df = session.sql("SELECT * FROM BANK_ANALYTICS.PUBLIC.VW_CREDIT_CARD_CHURN").to_pandas()
df.head()

In [ ]:
# drop columns that wont be used for prediction
df = df.drop(["ID", "CUSTOMER_NAME", "CREDIT_LIMIT", "FIRST_DATE_OPEN", "LAST_DATE_OPEN", "FIRST_EXP_DATE", 
              "LAST_EXP_DATE", "TOTAL_ACTIVE_MONTHS", "ACTIVE_YEARS", "AVG_TRX_PER_MONTH", 
              "TOTAL_BILLING_AMOUNT", "AVG_BILLING_AMOUNT", "LAST_TRX_DATE", "MONTHS_SINCE_LAST_TRX",
              "USAGE_PERCENT", "TRANSACTIONS_2025", "TOTAL_TRANSACTIONS"], axis=1)

# Drop rows with NaN values
df = df.dropna()

# list column to label encoded
columns_to_encode = ["GENDER", "MARITAL_STATUS"]

# Encode categorical variables except the target variable
label_encoders = {}
for column in columns_to_encode:
    label_encoders[column] = LabelEncoder()
    df[column] = label_encoders[column].fit_transform(df[column])

# Split data into features and targets
X = df.drop('CHURN_TARGET', axis=1)
y = df['CHURN_TARGET']

# Split data into Training and Testing Set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
## Train Random Forest Model
# Initialize Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators = 100, random_state = 42)

# Train the Model
rf_model .fit(X_train, y_train)

In [ ]:
## Evaluate Model
# Make Predictions
y_pred = rf_model.predict(X_test)

# Evaluate the Model
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassifier Report:")
print(classification_report(y_test, y_pred))

# Feature Selection Using Feature Importance
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

# Plot the Feature Importances
plt.figure(figsize=(8, 2))
sns.barplot(x=importances[indices], y=X.columns[indices])
plt.title("Feature Importances")
plt.xlabel("Relative Importances")
plt.ylabel("Feature Names")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

# Train XGBoost model
xgb_model = XGBClassifier(n_estimators=100, random_state=42,  eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# 1. Generate predictions using the trained XGBoost model
y_pred_xgb = xgb_model.predict(X_test)

# 2. Print out the performance summary
print(f"XGBoost Peak Accuracy: {accuracy_score(y_test, y_pred_xgb) * 100:.2f}%")
print("\nXGBoost Detailed Classification Report:")
print(classification_report(y_test, y_pred_xgb))

# 3. Pull feature importances from the model
xgb_importances = xgb_model.feature_importances_
xgb_indices = np.argsort(xgb_importances)[::-1]

# 4. Plot the visual bar chart
plt.figure(figsize=(8, 4))
sns.barplot(x=xgb_importances[xgb_indices], y=X_train.columns[xgb_indices], palette="viridis",hue=X_train.columns[xgb_indices])
plt.title("XGBoost - Feature Importances Breakdown", fontsize=14, pad=15)
plt.xlabel("Relative Importance Score", fontsize=12)
plt.ylabel("Feature Names", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# predict on new data
session = get_active_session()
session.sql("USE DATABASE BANK_ANALYTICS").collect()
session.sql("USE SCHEMA PUBLIC").collect()

new_query = session.sql("SELECT * FROM BANK_ANALYTICS.PUBLIC.VW_STAYED_CUSTOMERS").to_pandas()
new_query.head()

# retain the original dataframe to preserve unencoded columns
original_data = new_query.copy()

# retain original customerid column
customer_id = new_query['ID']

# drop columns that wont be used for predictions
new_query = new_query.drop(["ID", "CUSTOMER_NAME", "CREDIT_LIMIT", "FIRST_DATE_OPEN", "LAST_DATE_OPEN", "FIRST_EXP_DATE", 
              "LAST_EXP_DATE", "TOTAL_ACTIVE_MONTHS", "ACTIVE_YEARS", "AVG_TRX_PER_MONTH", 
              "TOTAL_BILLING_AMOUNT", "AVG_BILLING_AMOUNT", "LAST_TRX_DATE", "MONTHS_SINCE_LAST_TRX",
              "USAGE_PERCENT", "TRANSACTIONS_2025", "TOTAL_TRANSACTIONS", "CHURN_TARGET"], axis=1)

# encode categorical variables using the saved label encoders
for column in new_query.select_dtypes(include=['object']).columns:
    new_query[column] = label_encoders[column].transform(new_query[column])

# make predictions
new_predictions = rf_model.predict(new_query)

# add predictions to the original dataframe
original_data['Customer_Status_Pred'] = new_predictions

# filter the dataframe to include only records predicted as churned
original_data = original_data[original_data['Customer_Status_Pred'] == 1]

# convert original_data Pandas Dataframe into Snowflake Dataframe
snowspark_df = session.create_dataframe(original_data)

# save to Snowflake
snowspark_df.write.mode("overwrite").save_as_table("BANK_ANALYTICS.PUBLIC.PREDICTED_FUTURE_CHURNERS")

print("Table Created in Database")

In [ ]:
original_data